In [1]:
import torch
from visual_autolabel.image._model import HybridUNet
from pathlib import Path
import numpy as np

In [2]:
ckpt = torch.load("/data/visual-autolabel/models/best_hybrid_weights.pt",
                  map_location="cpu")

In [3]:
state = ckpt.get("state_dict", ckpt)

In [4]:
inputs3D = ('graymask', 'T1', 'T2')
inputs2D = ('curvature', 'convexity', 'thickness')
outputs = ('V1', 'V2', 'V3')

In [5]:
model = HybridUNet(len(inputs3D), len(inputs3D), len(inputs2D), len(outputs))

In [6]:
model.load_state_dict(state)

<All keys matched successfully>

In [7]:
import visual_autolabel as val

dataset2D_cache_path = '/data/visual-autolabel/datasets/HCP'
tx3Dto2D_cache_path = '/data/visual-autolabel/volumetric/data/tx3Dto2D'

val.image.dataset3D_cache_path = '/data/visual-autolabel/volumetric/data'
val.image.noddi_data_path = '/data/visual-autolabel/volumetric/NODDI'

In [8]:
class VolumeToFlatImageDataset(torch.utils.data.Dataset):
    def __init__(self,
                 sids,
                 cache_path=tx3Dto2D_cache_path):
        self.cache_path = Path(cache_path)
        self.sids = sids
        self.data = {}
    def __len__(self):
        return len(self.sids)
    def __getitem__(self, k):
        sid = self.sids[k]
        if sid in self.data:
            return self.data[sid]
        matrix = torch.load(f"{self.cache_path}/{sid}.pt", weights_only=False)
        import scipy.sparse as sps
        (row, col, val) = sps.find(matrix)
        matrix = torch.sparse_coo_tensor(
            torch.as_tensor(np.array([row, col])),
            torch.as_tensor(val),
            matrix.shape,
            dtype=torch.float32)
        #matrix = torch.tensor(matrix, dtype=torch.float32)
        self.data[sid] = matrix
        return matrix
        
class HCPHybridDataset(torch.utils.data.Dataset):
    def __init__(self,
                 sids,
                 inputs2D,
                 inputs3D,
                 outputs=('V1', 'V2', 'V3'),
                 cache_path2D=None,
                 cache_path3D=None,
                 transform_cache_path=tx3Dto2D_cache_path,
                 dtype=None,
                 device=None,
                 mkdir_mode=509,
                 subindex=(slice(2, -2, None), slice(8, 264, None), slice(2, -2, None)),
                 zoom=0.5,
                 ):
        self.transform_dataset = VolumeToFlatImageDataset(sids, cache_path=transform_cache_path)
        self.dataset3D = val.image.HCPDataset3D(
            sids=sids,
            inputs=inputs3D,
            outputs=outputs,
            cache_path=cache_path3D,
            dtype=dtype,
            device=device,
            mkdir_mode=0o775,
            subindex=subindex,
            zoom=zoom)
        self.dataset2D = val.benson2025.hcp.HCPDataset(
            inputs2D,
            outputs,
            sids=sids,
            cache_path=cache_path2D)
        self.sids = sids
    def __len__(self):
        return len(self.sids)
    def __getitem__(self, k):
        inputdata3D, _ = self.dataset3D[k]
        transformdata3D = self.transform_dataset[k]
        inputdata2D, outputdata2D = self.dataset2D[k]
        return (inputdata3D, transformdata3D, inputdata2D, outputdata2D)

def hybrid_collate(batch):
    inputs3D, transforms, inputs2D, labels = zip(*batch)
    return (
        torch.stack(inputs3D),   
        list(transforms),      
        torch.stack(inputs2D),  
        torch.stack(labels)      
    )

In [11]:
ds = HCPHybridDataset(
    sids=["100610"],
    inputs2D=inputs2D,              
    inputs3D=inputs3D,             
    outputs=outputs,                
    cache_path2D=dataset2D_cache_path,
    cache_path3D=val.image.dataset3D_cache_path,
    transform_cache_path=tx3Dto2D_cache_path,
)

data3D, sparse_tx, data2D, label = ds[0]

In [13]:
if data3D.dim()==4: 
    data3D = data3D.unsqueeze(0)
if data2D.dim()==3: 
    data2D = data2D.unsqueeze(0)

tx_list = [sparse_tx] 

device = next(model.parameters()).device
data3D  = data3D.to(device, dtype=torch.float32)
data2D  = data2D.to(device, dtype=torch.float32)
tx_list = [t.to(device) for t in tx_list]

with torch.no_grad():
    pred = model(data3D, data2D, tx_list)

print("pred:", pred.shape)


pred: torch.Size([1, 3, 128, 256])
